# EfficientAD-S on MVTec AD

This notebook is the Kaggle entry point. It uses the reusable `app.efficientad` modules, trains category-specific models, and never reads the local laptop path. MVTec AD remains subject to CC BY-NC-SA 4.0; keep the Kaggle dataset private and review the license before commercial use.

EfficientAD-S is implemented through the pinned Anomalib release. The notebook is an experiment runner, not a claim that training has already happened.

In [ ]:
%pip install -q -r /kaggle/working/requirements-kaggle-efficientad.txt
import os, sys, time, json, platform
from pathlib import Path
repo_candidates = [Path('/kaggle/working/repo'), Path('/kaggle/working'), Path.cwd()]
repo_root = next((p for p in repo_candidates if (p / 'app' / 'efficientad').is_dir()), None)
if repo_root is None: raise RuntimeError('Clone or upload this repository under /kaggle/working before running the notebook.')
sys.path.insert(0, str(repo_root))
import torch
print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available(), torch.version.cuda)
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))

In [ ]:
# MVTec AD is attached at the verified Kaggle input path below. This is a Kaggle-only path; it is not used by the local application.
from app.efficientad.data import discover_mvtec_manifest
dataset_root = Path('/kaggle/input/datasets/ipythonx/mvtec-ad')
if not dataset_root.is_dir():
    raise RuntimeError(f'MVTec AD dataset was not found at the configured Kaggle path: {dataset_root}')
if not (dataset_root / 'bottle' / 'train' / 'good').is_dir():
    raise RuntimeError(f'Invalid MVTec AD root; expected category directories below: {dataset_root}')
print('Dataset root:', dataset_root)
categories = ['bottle', 'screw', 'leather']
manifest = discover_mvtec_manifest(dataset_root, categories)
print('Categories:', manifest.categories)
print('Rows:', len(manifest.rows), 'Manifest SHA-256:', manifest.digest())
manifest_path = Path('/kaggle/working/efficientad_s_artifacts/dataset_manifest.json')
manifest.write(manifest_path)

## Training contract

The selected implementation is Anomalib 2.2.0 `EfficientAd(model_size='small')`. Teacher weights and any auxiliary data must be downloaded by the library only after reviewing their provenance and integrity. A failed or incomplete download is an error, never a randomly initialized substitute.

In [ ]:
from app.efficientad.config import EfficientADConfig
from app.efficientad.evaluation import evaluate_scores, select_threshold
config = EfficientADConfig(device='cuda' if torch.cuda.is_available() else 'cpu', seed=42, epochs=100)
print(json.dumps(config.as_dict(), indent=2))
print('Config SHA-256:', config.digest())

In [ ]:
# Reference Anomalib training call. Keep one model per category and save its checkpoint outside Git.
from anomalib.data import MVTec
from anomalib.engine import Engine
from anomalib.models import EfficientAd

artifact_root = Path('/kaggle/working/efficientad_s_artifacts')
artifact_root.mkdir(parents=True, exist_ok=True)
training_summary = []
for category in categories:
    started = time.time()
    datamodule = MVTec(root=str(dataset_root), category=category, image_size=config.image_size, train_batch_size=config.batch_size, eval_batch_size=config.batch_size)
    model = EfficientAd(model_size='small')
    engine = Engine(default_root_dir=str(artifact_root / category))
    engine.fit(model=model, datamodule=datamodule)
    engine.test(model=model, datamodule=datamodule)
    training_summary.append({'category': category, 'duration_seconds': time.time() - started})
print(json.dumps(training_summary, indent=2))

## Export and qualification boundary

Before calling a model qualified, add the actual validation threshold, test metrics, checkpoint SHA-256, preprocessing settings, dependency versions, and training hardware to a metadata file. Do not tune thresholds on the final test split. Do not report metrics from a partial or failed category run.

In [ ]:
# Record the run envelope; replace placeholders only with measured values from the completed run.
run_metadata = {
  'experiment': 'efficientad-s-mvtec-v1',
  'categories_requested': categories,
  'dataset_manifest_sha256': manifest.digest(),
  'config_sha256': config.digest(),
  'license': 'CC BY-NC-SA 4.0',
  'hardware': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
  'training_summary': training_summary,
  'qualification_status': 'NOT_QUALIFIED_UNTIL_METRICS_AND_ARTIFACT_HASHES_ARE_RECORDED'
}
(artifact_root / 'run_metadata.json').write_text(json.dumps(run_metadata, indent=2) + '\n')
print(json.dumps(run_metadata, indent=2))